## Machine Learning Pipeline

As the class practice, the students will be required to develop a machine learning pipeline using the `Churn_Modelling_train_test.csv` dataset.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)


### Exploratory Data Analysis

In this section the students are required to do an EDA to understand the dataset.

In [1]:
# import packages
import pandas as pd
import joblib
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

In [3]:
# Load the dataset
df = pd.read_csv("Churn_Modelling_train_test.csv")

In [4]:
# dataset analysis
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,4784,15729224,Jennings,710,France,Female,37.0,5,0.00,2,1.0,0.0,115403.31,0
1,1497,15799156,Okwuadigbo,569,Spain,Male,38.0,8,0.00,2,0.0,0.0,79618.79,0
2,1958,15674922,Beavers,710,France,Male,54.0,6,171137.62,1,1.0,1.0,167023.95,1
3,9174,15653572,Thornton,673,Spain,Male,43.0,8,127132.96,1,0.0,1.0,6009.27,1
4,9748,15775761,Iweobiegbunam,610,Germany,Female,69.0,5,86038.21,3,0.0,0.0,192743.06,1


In [14]:
# Churn Dataset Cleaning and Correlation-Ready Pipeline
# - Clean dataset for machine learning and analytics
# - Prepare correlation-ready numerical dataset
# - Handle missing values, duplicates, outliers, encoding
# - Generate engineered features
# - Export cleaned datasets
# Strip spaces from column names immediately
df.columns = df.columns.str.strip()
# Remove duplicates
df = df.drop_duplicates()
# Standardize datatypes
# Convert binary columns
binary_cols = ['HasCrCard', 'IsActiveMember', 'Exited']

for col in binary_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# Missing values:
# - Numerical -> median
# - Categorical -> most frequent
numerical_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Numerical imputation
num_imputer = SimpleImputer(strategy='median')
df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])

# Categorical imputation
cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

# Clean string val
for col in categorical_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.title()
    )

# IQR for clipping outliers
numeric_features = [
    col for col in numerical_cols
    if col != 'Exited'
]

for col in numeric_features:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[col] = np.clip(df[col], lower_bound, upper_bound)


In [ ]:
# Create engineered variables in preparation for correlation analysis
# Balance-to-Salary Ratio
if {'Balance', 'EstimatedSalary'}.issubset(df.columns):
    df['BalanceSalaryRatio'] = (
        df['Balance'] / (df['EstimatedSalary'] + 1)
    )

# Age Groups
if 'Age' in df.columns:
    df['AgeGroup'] = pd.cut(
        df['Age'],
        bins=[0, 25, 35, 50, 65, 100],
        labels=['18-25', '26-35', '36-50', '51-65', '65+']
    )

# Product Engagement Score
if {'NumOfProducts', 'IsActiveMember'}.issubset(df.columns):
    df['EngagementScore'] = (
        df['NumOfProducts'] * df['IsActiveMember']
    )

# Copy before encoding
clean_df = df.copy()

# One-hot encoding for nominal categories
encoded_df = pd.get_dummies(
    clean_df,
    columns=['Geography', 'Gender', 'AgeGroup'],
    drop_first=True
)

# Try correlation
correlation_matrix = encoded_df.corr(numeric_only=True)

# Sort by correlation with target variable
if 'Exited' in correlation_matrix.columns:

    churn_corr = (
        correlation_matrix['Exited']
        .sort_values(ascending=False)
    )

    print('\n================ CORRELATION WITH CHURN ================')
    print(churn_corr)


# Export the clean dataset
# Main  dataset
encoded_df.to_csv(
    'cleaned_churn_dataset.csv',
    index=False
)

# Correlation matrix
correlation_matrix.to_csv(
    'churn_correlation_matrix.csv'
)

# Descriptive analysis, Full statistical summary
stats_summary = encoded_df.describe(include='all').transpose()
print(stats_summary)

# Additional advanced metrics
advanced_stats = pd.DataFrame({
    'mean': encoded_df.mean(numeric_only=True),
    'median': encoded_df.median(numeric_only=True),
    'std_dev': encoded_df.std(numeric_only=True),
    'variance': encoded_df.var(numeric_only=True),
    'min': encoded_df.min(numeric_only=True),
    'max': encoded_df.max(numeric_only=True),
    'skewness': encoded_df.skew(numeric_only=True),
    'kurtosis': encoded_df.kurtosis(numeric_only=True)
})


================ CORRELATION WITH CHURN ================
Exited                1.000000
Age                   0.309255
AgeGroup_51-65        0.223137
Geography_Germany     0.175439
Balance               0.116550
AgeGroup_36-50        0.103516
BalanceSalaryRatio    0.027448
EstimatedSalary       0.010752
CustomerId           -0.006444
HasCrCard            -0.010679
Tenure               -0.013248
RowNumber            -0.019545
CreditScore          -0.024016
Geography_Spain      -0.053332
NumOfProducts        -0.063161
Gender_Male          -0.107566
EngagementScore      -0.145120
IsActiveMember       -0.159237
AgeGroup_26-35       -0.223402
AgeGroup_65+               NaN
Name: Exited, dtype: float64
                     count unique    top  freq             mean           std  \
RowNumber           8999.0    NaN    NaN   NaN       5002.25614   2884.146186   
CustomerId          8999.0    NaN    NaN   NaN  15690732.526058   71798.44394   
Surname               8999   2771  Smith    29    

In [18]:
print('\n================ DESCRIPTIVE STATS ================')
print(advanced_stats)


================ DESCRIPTIVE STATS ================
                            mean        median       std_dev      variance  \
RowNumber           5.002256e+03  4.988000e+03   2884.146186  8.318299e+06   
CustomerId          1.569073e+07  1.569100e+07  71798.443940  5.155017e+09   
CreditScore         6.506811e+02  6.520000e+02     96.446311  9.301891e+03   
Age                 3.864619e+01  3.700000e+01      9.722342  9.452394e+01   
Tenure              5.008223e+00  5.000000e+00      2.894250  8.376683e+00   
Balance             7.621635e+04  9.699709e+04  62436.547243  3.898322e+09   
NumOfProducts       1.528725e+00  1.000000e+00      0.568670  3.233857e-01   
HasCrCard           7.056340e-01  1.000000e+00      0.455783  2.077378e-01   
IsActiveMember      5.148350e-01  1.000000e+00      0.499808  2.498077e-01   
EstimatedSalary     1.001885e+05  1.005570e+05  57563.825037  3.313594e+09   
Exited              2.043560e-01  0.000000e+00      0.403253  1.626127e-01   
BalanceSala

In [19]:
# review categorical variables
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print("Categorical Variables:")
print(categorical_cols)

# Display unique values and counts
for col in categorical_cols:
    print(f"\n================ {col.upper()} ================")
    print(df[col].value_counts(dropna=False))

Categorical Variables:
['Surname', 'Geography', 'Gender']

================ SURNAME ================
Surname
Smith          29
Martin         25
Yeh            25
Walker         23
Shih           23
               ..
Beit            1
Onwuamaegbu     1
Edith           1
Holder          1
Macrossan       1
Name: count, Length: 2771, dtype: int64

================ GEOGRAPHY ================
Geography
France     4505
Spain      2250
Germany    2244
Name: count, dtype: int64

================ GENDER ================
Gender
Male      4905
Female    4094
Name: count, dtype: int64


In [20]:
# review binary variables

# Identify binary columns
binary_cols = []

for col in df.columns:
    if df[col].nunique() == 2:
        binary_cols.append(col)

print("Binary Variables:")
print(binary_cols)

# Display binary distributions
for col in binary_cols:
    print(f"\n================ {col.upper()} ================")
    print(df[col].value_counts(dropna=False))

Binary Variables:
['Gender', 'HasCrCard', 'IsActiveMember', 'Exited']

================ GENDER ================
Gender
Male      4905
Female    4094
Name: count, dtype: int64

================ HASCRCARD ================
HasCrCard
1.0    6350
0.0    2649
Name: count, dtype: int64

================ ISACTIVEMEMBER ================
IsActiveMember
1.0    4633
0.0    4366
Name: count, dtype: int64

================ EXITED ================
Exited
0.0    7160
1.0    1839
Name: count, dtype: int64


In [21]:
# review numerical variables

# Identify numerical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Numerical Variables:")
print(numerical_cols)

# Display descriptive statistics
print("\n================ NUMERICAL SUMMARY ================")
print(df[numerical_cols].describe().transpose())

# Additional statistics
for col in numerical_cols:
    print(f"\n================ {col.upper()} ================")
    print(f"Mean: {df[col].mean()}")
    print(f"Median: {df[col].median()}")
    print(f"Standard Deviation: {df[col].std()}")
    print(f"Variance: {df[col].var()}")
    print(f"Minimum: {df[col].min()}")
    print(f"Maximum: {df[col].max()}")
    print(f"Skewness: {df[col].skew()}")
    print(f"Kurtosis: {df[col].kurtosis()}")

Numerical Variables:
['RowNumber', 'CustomerId', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'BalanceSalaryRatio', 'EngagementScore']

================ NUMERICAL SUMMARY ================
                     count          mean           std          min  \
RowNumber           8999.0  5.002256e+03   2884.146186         2.00   
CustomerId          8999.0  1.569073e+07  71798.443940  15565701.00   
CreditScore         8999.0  6.506811e+02     96.446311       383.00   
Age                 8999.0  3.864619e+01      9.722342        18.00   
Tenure              8999.0  5.008223e+00      2.894250         0.00   
Balance             8999.0  7.621635e+04  62436.547243         0.00   
NumOfProducts       8999.0  1.528725e+00      0.568670         1.00   
HasCrCard           8999.0  7.056340e-01      0.455783         0.00   
IsActiveMember      8999.0  5.148350e-01      0.499808         0.00   
EstimatedSalary     8999.0 

**EDA Conclusions**

From this Exploratory Data Analysis we can retrieve the following information:
* *Customer Age vs Churn Behavior*
* *Customer Activity and Engagement*
* *Number of Banking products availed*
* *Figure out which country has the highest churning*
* *Account Balance and Credit Scores*
* *Credit Card Ownership and customer tenure*

Note: we can focus on things that could be related to churning


### ML Pipeline

In this section the students are required to create a ML pipeline to predict whether the customer left the bank or not.

In [108]:
# import packages, general ones
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from collections import Counter

# Models
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [109]:
# Load the dataset
df = pd.read_csv("cleaned_churn_dataset.csv")

In [110]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,...,Exited,BalanceSalaryRatio,EngagementScore,Geography_Germany,Geography_Spain,Gender_Male,AgeGroup_26-35,AgeGroup_36-50,AgeGroup_51-65,AgeGroup_65+
0,4784.0,15729224.0,Jennings,710.0,37.0,5.0,0.00,2.0,1.0,0.0,...,0.0,0.000000,0.0,False,False,False,False,True,False,False
1,1497.0,15799156.0,Okwuadigbo,569.0,38.0,8.0,0.00,2.0,0.0,0.0,...,0.0,0.000000,0.0,False,True,True,False,True,False,False
2,1958.0,15674922.0,Beavers,710.0,54.0,6.0,171137.62,1.0,1.0,1.0,...,1.0,1.024623,1.0,False,False,True,False,False,True,False
3,9174.0,15653572.0,Thornton,673.0,43.0,8.0,127132.96,1.0,0.0,1.0,...,1.0,21.152620,1.0,False,True,True,False,True,False,False
4,9748.0,15775761.0,Iweobiegbunam,610.0,62.0,5.0,86038.21,3.0,0.0,0.0,...,1.0,0.446386,0.0,True,False,False,False,False,True,False


In [111]:
# Check shape
print(df.shape)

(8999, 21)


In [112]:
# Features
X = df.drop('Exited', axis=1)

# Labels
y = df['Exited']

print("Features Shape:", X.shape)
print("Labels Shape:", y.shape)

Features Shape: (8999, 20)
Labels Shape: (8999,)


In [113]:
# initialize scaler globally
scaler = StandardScaler()

# Preprocess the features - create a function or a class for it 
def transform(df: pd.DataFrame) -> pd.DataFrame:

    # create copy
    df = df.copy()

    # ensure all columns are numeric
    df = df.apply(pd.to_numeric, errors='coerce')

    # fill remaining unexpected nulls
    df = df.fillna(0)

    # scale features
    scaled_array = scaler.fit_transform(df)

    # convert back to dataframe
    df_scaled = pd.DataFrame(
        scaled_array,
        columns=df.columns,
        index=df.index
    )

    return df_scaled

In [114]:
# apply the transform function
X_processed = transform(X)

print(X_processed.head())

   RowNumber  CustomerId  Surname  CreditScore       Age    Tenure   Balance  \
0  -0.075679    0.536134      0.0     0.615080 -0.169329 -0.002841 -1.220769   
1  -1.215421    1.510193      0.0    -0.846954 -0.066468  1.033754 -1.220769   
2  -1.055572   -0.220219      0.0     0.615080  1.579318  0.342691  1.520368   
3   1.446520   -0.517596      0.0     0.231426  0.447840  1.033754  0.815539   
4   1.645550    1.184332      0.0    -0.421824  2.402210 -0.002841  0.157318   

   NumOfProducts  HasCrCard  IsActiveMember  EstimatedSalary  \
0       0.828777   0.645883       -1.030123         0.264326   
1       0.828777  -1.548267       -1.030123        -0.357358   
2      -0.929809   0.645883        0.970757         1.161131   
3      -0.929809  -1.548267        0.970757        -1.636175   
4       2.587364  -1.548267       -1.030123         1.607949   

   BalanceSalaryRatio  EngagementScore  Geography_Germany  Geography_Spain  \
0           -0.036998        -0.912493          -0.57636

In [115]:
from sklearn.model_selection import train_test_split

# 80-20 split using processed features
X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# verify shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (7199, 20)
X_test: (1800, 20)
y_train: (7199,)
y_test: (1800,)


In [116]:
# balance the dataset (if you think it is necessary)
# initialize SMOTE
smote = SMOTE(random_state=42)

# balance only training dataset
X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train,
    y_train
)

# class distribution check
print("Before SMOTE:")
print(Counter(y_train))

print("\nAfter SMOTE:")
print(Counter(y_train_balanced))

Before SMOTE:
Counter({0.0: 5728, 1.0: 1471})

After SMOTE:
Counter({0.0: 5728, 1.0: 5728})


In [ ]:
# Train a decision tree model using the `train sample`
from sklearn.ensemble import RandomForestClassifier

# initialize random forest model
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

# train model
model.fit(X_train_balanced, y_train_balanced)

print("Random Forest Training Complete") # Best performance so far

Random Forest Training Complete


In [ ]:
# from sklearn.linear_model import LogisticRegression 
# initialize model 
# model = LogisticRegression(max_iter=1000) 
# train model using balanced data 
# model.fit(X_train_balanced, y_train_balanced)

# Logistic regression's ROC AUC: 0.7667294146222978

In [118]:
# Check the accuracy of the model on the `test sample`
# make predictions on untouched test set

# predictions
y_pred = model.predict(X_test)

# probabilities
y_prob = model.predict_proba(X_test)[:, 1]

print("Prediction Complete")

Prediction Complete


In [ ]:
# Analyze performance
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8327777777777777
Precision: 0.5831265508684863
Recall: 0.6385869565217391
F1 Score: 0.6095979247730221
ROC AUC: 0.8431579426767063

Classification Report
              precision    recall  f1-score   support

         0.0       0.90      0.88      0.89      1432
         1.0       0.58      0.64      0.61       368

    accuracy                           0.83      1800
   macro avg       0.74      0.76      0.75      1800
weighted avg       0.84      0.83      0.84      1800


Confusion Matrix
[[1264  168]
 [ 133  235]]


### MLFlow Integration

In this section the students are required to track the model and the parameters into MLFlow.

In [127]:
# import packages for mlfow
import mlflow
from mlflow.models import infer_signature
print(mlflow.__version__)

3.12.0


Start a local Tracking server on port 8080 - it is necessary to import mlflow and open a new terminal

In [128]:
# Set our tracking server uri for logging on port 8080
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")
print(mlflow.get_tracking_uri())

http://127.0.0.1:8080


In [129]:
# mlflow.set_tracking_uri("file:./mlruns")
# print(mlflow.get_tracking_uri())

Visit the MLFlow UI on `http://127.0.0.1:8080`

In [130]:
# Create a new MLflow Experiment called `Practice Experiment - {Your Name}`
mlflow.set_experiment("Practice Experiment - Mark Joshua Ponciano Halili")

2026/05/15 18:06:53 INFO mlflow.tracking.fluent: Experiment with name 'Practice Experiment - Mark Joshua Ponciano Halili' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/mjhalili/Desktop/EADA_Files/ML_Operations/mlops-and-systems-design/session_2/Class_Exercise/mlruns/1', creation_time=1778861213431, experiment_id='1', last_update_time=1778861213431, lifecycle_stage='active', name='Practice Experiment - Mark Joshua Ponciano Halili', tags={}, trace_location=None, workspace='default'>

In [131]:
# define parameters manually
params = {
    "model_type": "RandomForestClassifier",
    "n_estimators": 300,
    "max_depth": 12,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "random_state": 42
}

In [133]:
f1 = f1_score(y_test, y_pred)

In [134]:
auc_halili = roc_auc_score(y_test, y_prob)

In [ ]:
with mlflow.start_run():

    mlflow.log_params(params)

    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("precision", precision_score(y_test, y_pred))
    mlflow.log_metric("recall", recall_score(y_test, y_pred))
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("AUC_ROC", auc_halili)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Session 2 Class Exercise", "Data Preparation, Model Training and Prediction")

🏃 View run entertaining-asp-342 at: http://127.0.0.1:8080/#/experiments/1/runs/8bd377fec6b342ce9e203540f48775ae
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


In [136]:
# Start an MLflow run

with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("f1_score", f1)

    # Log the AUC ROC
    mlflow.log_metric("AUC ROC", auc_halili)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Session 2 Class Exercise", "Data Preparation, Model Training and Prediction")

    # infer model signature
    signature = infer_signature(X_train, model.predict(X_train))

    # log the model
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="random_forest_model",
        signature=signature,
        input_example=X_train.iloc[:5]
    )



2026/05/15 18:11:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 18:11:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run carefree-shark-876 at: http://127.0.0.1:8080/#/experiments/1/runs/2b66137b5d3e43f887096749328d3d92
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


### Keep practicing

In [ ]:
# Create a new experiment using other models or different parameters
# initialize XGBoost model
# model = XGBClassifier(
    # n_estimators=500,
    # max_depth=6,
    # learning_rate=0.03,
    # subsample=0.8,
    # colsample_bytree=0.8,
    # objective='binary:logistic',
    # eval_metric='auc',
    # random_state=42
# )

# train model
# model.fit(
    # X_train_balanced,
    # y_train_balanced
# )

# XGBoost's ROC AUC: 0.8400467952999758